# MedStream - Clinical Dashboard & Prediction

Clinical monitoring dashboard and real-time diabetes risk assessment:
1. Queries key metrics (KPIs) from the DuckDB analytical warehouse (`patients_analytics`).
2. Visualizes age, glucose, and HbA1c distributions by diagnosis.
3. Evaluates patient profiles in real-time using the trained Random Forest model.

### 1. Warehouse Clinical Overview

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect('medstream_warehouse.duckdb', read_only=True)
df = con.execute("SELECT * FROM patients_analytics").df()
con.close()

total_pts = len(df)
diabetic_pts = int((df['diabetes'] == 1).sum())
diabetic_pct = (diabetic_pts / total_pts) * 100

print("=" * 45)
print("         MEDSTREAM CLINICAL KPIs")
print("=" * 45)
print(f"Total Cohort:        {total_pts:,} patients")
print(f"Diagnosed Cases:     {diabetic_pts:,} ({diabetic_pct:.2f}%)")
print(f"Mean Fasting Sugar:  {df['blood_glucose_level'].mean():.1f} mg/dL")
print(f"Mean HbA1c Level:    {df['HbA1c_level'].mean():.2f}")
print(f"Mean Patient BMI:    {df['bmi'].mean():.1f}")
print("=" * 45)

### 2. Clinical Indicators by Diagnosis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# age
df.boxplot(column='age', by='diabetes', ax=axes[0])
axes[0].set_title('Age by Diagnosis')
axes[0].set_xlabel('Diabetes (0 = No, 1 = Yes)')
axes[0].set_ylabel('Age (years)')

# glucose
df.boxplot(column='blood_glucose_level', by='diabetes', ax=axes[1])
axes[1].set_title('Glucose by Diagnosis')
axes[1].set_xlabel('Diabetes (0 = No, 1 = Yes)')
axes[1].set_ylabel('Glucose (mg/dL)')

# HbA1c
df.boxplot(column='HbA1c_level', by='diabetes', ax=axes[2])
axes[2].set_title('HbA1c by Diagnosis')
axes[2].set_xlabel('Diabetes (0 = No, 1 = Yes)')
axes[2].set_ylabel('HbA1c Level')

plt.suptitle('')
plt.tight_layout()
plt.show()

### 3. Load Trained Model

In [ ]:
import joblib

model = joblib.load('models/diabetes_model.pkl')
feature_cols = joblib.load('models/model_columns.pkl')
print(f"Model loaded successfully ({len(feature_cols)} features)")

### 4. Real-Time Clinical Risk Assessment
Define a patient's clinical parameters to run real-time risk assessment.

In [ ]:
def assess_patient(age, gender, hypertension, heart_disease, bmi, hba1c, glucose, smoking='never'):
    patient_dict = {
        'age': age,
        'hypertension': hypertension,
        'heart_disease': heart_disease,
        'bmi': bmi,
        'HbA1c_level': hba1c,
        'blood_glucose_level': glucose,
        'gender_code': 1 if gender == 'Male' else 0,
        'gender_Male': 1 if gender == 'Male' else 0,
        'smoking_history_former': 1 if smoking == 'former' else 0,
        'smoking_history_never': 1 if smoking == 'never' else 0,
        'smoking_history_No Info': 1 if smoking == 'No Info' else 0
    }
    
    patient_df = pd.DataFrame([patient_dict])
    for col in feature_cols:
        if col not in patient_df.columns:
            patient_df[col] = 0
    patient_df = patient_df[feature_cols]
    
    prob = model.predict_proba(patient_df)[0][1] * 100
    pred = model.predict(patient_df)[0]
    
    print("-" * 45)
    print(f"PATIENT PROFILE: Age {age} | {gender} | BMI {bmi}")
    print(f"LAB TESTS:       HbA1c {hba1c} | Glucose {glucose} mg/dL")
    print("-" * 45)
    if pred == 1:
        print(f"RESULT: [!] HIGH RISK ({prob:.1f}% probability)")
        print("RECOMMENDATION: Patient requires clinical diabetic evaluation.")
    else:
        print(f"RESULT: [+] LOW RISK ({prob:.1f}% probability)")
        print("RECOMMENDATION: Patient parameters fall within normal ranges.")
    print("-" * 45)

# Test Case 1: Healthy Profile
assess_patient(age=30, gender='Female', hypertension=0, heart_disease=0, bmi=22.5, hba1c=5.2, glucose=95)

# Test Case 2: High Risk Profile
assess_patient(age=62, gender='Male', hypertension=1, heart_disease=1, bmi=33.0, hba1c=7.8, glucose=210)